# EXP-G-001, attempt 5: the registered (LLM-assisted) extractor

**Its own session.** Accelerator **GPU T4 x2**, Internet **on**. No dataset, no detector,
no SHAP, no generator tokens: the 300 audit texts are already in the repo.

What it does:
1. clones the repo at `PINNED_REF`;
2. downloads Google's QAT Q4_0 GGUF of `gemma-4-26B-A4B-it` (14.4 GB) at the pinned
   revision and checks its sha256;
3. serves it with Ollama, using Gemma 4's chat format with thinking off;
4. re-parses the audit texts and, when `LIMIT = '0'`, scores the gate, plus the
   score without the 12 smoke-test items (amendment 0006; a check, not the verdict).

Why Ollama and not transformers: see amendment 0005
(`configs/statistics/amendments/0005-extractor-weights-qat-gguf.md`). In short,
bitsandbytes cannot shrink Gemma 4's experts, so the old load needed ~48 GB.

**Smoke first** (`LIMIT = '12'`). Read `reextract.log`: if the evidence counts show
`llm` claims and few items fell back to the rules, run again with `LIMIT = '0'`.

Everything worth keeping lands in `/kaggle/working/section11_artifacts.zip`.

In [ ]:
# Launcher parameters ONLY. All logic lives in the repo.
import os
PINNED_REF = 'main'   # the terminal push replaces this with the exact commit
LIMIT      = '12'     # '12' = smoke (minutes) | '0' = all 300 texts, then score the gate

os.environ['FAITHFULIDS_REF'] = PINNED_REF
os.environ['FAITHFULIDS_EXTRACTION_LIMIT'] = LIMIT
OUT = '/kaggle/working/section11'
os.makedirs(OUT, exist_ok=True)
print('ref:', PINNED_REF, '| limit:', LIMIT, '(0 = all 300 + gate)')

In [ ]:
%%bash
set -e
set -o pipefail   # each pip line pipes into `tail`: without this a failed install passes silently
nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || {
  echo "NO GPU. Settings -> Accelerator -> 'GPU T4 x2'."; exit 1; }
GPUS=$(nvidia-smi --list-gpus | wc -l)
if [ "$GPUS" -lt 2 ]; then
  echo "WARNING: only $GPUS GPU. The model needs both T4s to stay fully on GPU;"
  echo "Ollama will put the rest on CPU. Slower, still correct."
fi
# Clone OUTSIDE /kaggle/working, so the saved output is only what section11/ collects.
rm -rf /tmp/faithful-ids
git clone --quiet https://github.com/SLIMIHamda/faithful-ids.git /tmp/faithful-ids
cd /tmp/faithful-ids && git checkout --quiet "$FAITHFULIDS_REF"
git rev-parse HEAD | tee /kaggle/working/section11/PINNED_COMMIT.txt
# --ignore-requires-python: Kaggle runs Python 3.13; pyproject's `<3.13` is for the pinned
# stack, which --no-deps already skips. The import below proves the install took.
pip install -q -e . --no-deps --ignore-requires-python 2>&1 | tail -n 2
pip install -q pyyaml jsonschema 2>&1 | tail -n 2
python -c "import faithfulids, sys; print('faithfulids', faithfulids.__version__, 'installed | python', sys.version.split()[0])"

In [ ]:
# Download the pinned GGUF and prove it is the pinned file. Pins come from the repo config.
import hashlib, time, yaml
from huggingface_hub import hf_hub_download

m = yaml.safe_load(open('/tmp/faithful-ids/configs/extraction/eval_extractor.yaml'))['model']
w, ol = m['weights'], m['ollama']
t0 = time.time()
GGUF = hf_hub_download(w['hf_repo'], w['file'], revision=w['revision'], local_dir='/tmp/gguf')
print(f"downloaded {os.path.getsize(GGUF) / 2**30:.2f} GiB in {(time.time() - t0) / 60:.1f} min")
h = hashlib.sha256()
with open(GGUF, 'rb') as f:
    for chunk in iter(lambda: f.read(1 << 24), b''):
        h.update(chunk)
assert h.hexdigest() == w['sha256'], f"sha256 {h.hexdigest()} is not the pinned {w['sha256']}"
print('sha256 matches the pin:', w['sha256'])
with open('/tmp/gguf/Modelfile', 'w') as f:
    f.write(f"FROM {GGUF}\nRENDERER {ol['renderer']}\nPARSER {ol['parser']}\n")

In [ ]:
# Install Ollama, start it, and load the pinned file under the config's model name.
import json, subprocess, urllib.request

subprocess.run('apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y zstd >/dev/null 2>&1', shell=True)
subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)

env = dict(os.environ, OLLAMA_HOST='127.0.0.1:11434', OLLAMA_KEEP_ALIVE='-1', OLLAMA_NUM_PARALLEL='1')
serve = subprocess.Popen(['ollama', 'serve'], env=env, stderr=subprocess.STDOUT,
                         stdout=open(f'{OUT}/ollama_serve.log', 'w'))
for _ in range(60):
    try:
        ver = json.load(urllib.request.urlopen('http://127.0.0.1:11434/api/version', timeout=2))['version']
        break
    except Exception:
        time.sleep(2)
else:
    raise SystemExit('ollama serve did not start: see section11/ollama_serve.log')
need = tuple(int(x) for x in ol['min_version'].split('.'))
have = tuple(int(x) for x in ver.split('-')[0].split('.')[:3])
assert have >= need, f"Ollama {ver} is older than {ol['min_version']}, which Gemma 4 needs"
open(f'{OUT}/OLLAMA_VERSION.txt', 'w').write(ver + '\n')
print('ollama', ver)

subprocess.run(['ollama', 'create', ol['model_name'], '-f', '/tmp/gguf/Modelfile'], check=True)
os.remove(GGUF)  # Ollama now holds its own copy, stored under the same sha256

# Warm-up: load the model once and show where its layers went (GPU vs CPU).
t0 = time.time()
req = urllib.request.Request('http://127.0.0.1:11434/api/chat', method='POST',
    headers={'Content-Type': 'application/json'},
    data=json.dumps({'model': ol['model_name'], 'stream': False, 'think': False,
                     'messages': [{'role': 'user', 'content': 'Reply with the single word OK.'}],
                     'options': {'temperature': 0, 'seed': 0, 'num_predict': 8, 'num_ctx': 8192}}).encode())
reply = json.load(urllib.request.urlopen(req, timeout=1800))
print(f"warm-up reply {reply['message']['content']!r} after {time.time() - t0:.0f}s")
print(subprocess.run(['ollama', 'ps'], capture_output=True, text=True).stdout)

In [ ]:
%%bash
# Re-extract, then (LIMIT=0 only) score the gate. Logs and artifacts are collected
# even when a step fails; the cell still fails afterwards so the run shows as an error.
cd /tmp/faithful-ids
export PYTHONPATH=src
OUT=/kaggle/working/section11
B=experiments/gates/EXP-G-001_audit_v2
LIMIT="${FAITHFULIDS_EXTRACTION_LIMIT:-12}"
ARGS="--batch $B"
if [ "$LIMIT" != "0" ]; then ARGS="$ARGS --limit $LIMIT"; fi
set -o pipefail

python tools/reextract_llm_assisted.py $ARGS 2>&1 | tee $OUT/reextract.log
rc=$?
if [ $rc -eq 0 ] && [ "$LIMIT" = "0" ]; then
  KEY=$(python -c "import yaml; v = yaml.safe_load(open('configs/extraction/eval_extractor.yaml'))['version']; print('extractor_claims_' + v.replace('.', '_') + '_llm')")
  echo "== EXP-G-001, attempt 5: the registered instrument ($KEY) =="
  python tools/score_audit_gate.py --batch $B --pass LLM_1_V2 --pass LLM_2_V2 \
         --claims-key "$KEY" 2>&1 | tee $OUT/score.log
  rc=$?
  if [ $rc -eq 0 ]; then
    cp $B/gate_result.json $OUT/
    echo "== sensitivity (amendment 0006): without the 12 smoke-test items, NOT the verdict =="
    python tools/score_audit_gate.py --batch $B --pass LLM_1_V2 --pass LLM_2_V2 \
           --claims-key "$KEY" --exclude-first 12 2>&1 | tee $OUT/score_excl_first12.log
    cp $B/gate_result_excl_first12.json $OUT/ 2>/dev/null
  fi
elif [ $rc -eq 0 ]; then
  echo "SMOKE ONLY ($LIMIT items): the gate is not scored. Next run: LIMIT='0'."
fi

cp $B/audit_key_DO_NOT_SHOW_ANNOTATOR.json $OUT/ 2>/dev/null
cp -r $B/_llm_extraction_cache $OUT/ 2>/dev/null
nvidia-smi > $OUT/nvidia-smi.txt 2>&1
cd /kaggle/working && rm -f section11_artifacts.zip && zip -qr section11_artifacts.zip section11
ls -la section11_artifacts.zip
exit $rc